# Hate Speech Classification with Robustness Testing
**Dataset:** HateXplain (Mathew et al., 2021)  
**Model:** DistilBERT fine-tuned for 3-class classification (hate / offensive / normal)  
**Focus:** Robustness to text obfuscation — leet-speak, punctuation insertion, character repetition

---

## ⚙️ Configuration

All hyperparameters and paths are set here. Edit this cell to change the experiment.

In [ ]:
# ── Install dependencies (run once, then Runtime → Restart session) ──
# !pip install transformers torch scikit-learn matplotlib seaborn wordcloud nltk --quiet

# ── Configuration — edit here ────────────────────────────
MODEL_NAME   = "distilbert-base-uncased"   # change to "GroNLP/hateBERT" to use hateBERT
MAX_LEN      = 128                  # max token length for tokenizer
BATCH_SIZE   = 16                   # reduce to 8 if OOM on Colab free tier
LR           = 2e-5                 # learning rate
WEIGHT_DECAY = 0.1                  # L2 regularization
MAX_EPOCHS   = 8                    # upper bound — early stopping will cut this short
PATIENCE     = 2                    # early stopping patience (epochs without val loss improvement)
AUG_RATE     = 0.5                  # probability of augmenting hate/offensive posts in 4b
RANDOM_SEED  = 42

# ── Paths ─────────────────────────────────────────────────
DRIVE_DIR    = "/content/drive/MyDrive/hate_speech_project"  # where to save models
MODEL_SAVE   = "distilbert_baseline"
MODEL_IMP_SAVE = "distilbert_improved"

# ── Label mapping ─────────────────────────────────────────
LABEL_MAP    = {"hatespeech": 0, "offensive": 1, "normal": 2}
LABEL_NAMES  = {0: "hate", 1: "offensive", 2: "normal"}
COLORS       = {"hate": "#e63946", "offensive": "#f4a261", "normal": "#2a9d8f"}

## 📦 Imports & Shared Utilities

All imports and shared functions (dataset class, training loop, obfuscation) are defined once here and reused across tasks.

In [ ]:
import json, re, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if device.type == "cuda":
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Shared: PyTorch Dataset ───────────────────────────────
class HateDataset(Dataset):
    """
    Tokenizes text on-the-fly. Optionally applies an obfuscation
    function to hate/offensive posts during training (adversarial augmentation).
    """
    def __init__(self, texts, labels, tokenizer, augment_fn=None, aug_rate=0.5):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.augment_fn = augment_fn
        self.aug_rate   = aug_rate

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text  = self.texts[idx]
        label = self.labels[idx]
        if self.augment_fn and label in [0, 1] and random.random() < self.aug_rate:
            text = self.augment_fn(text)
        enc = self.tokenizer(text, max_length=MAX_LEN, padding="max_length",
                             truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label":          torch.tensor(label, dtype=torch.long)}

In [ ]:
# ── Shared: evaluate() ────────────────────────────────────
def evaluate(model, loader, criterion=None):
    """Returns macro F1, avg loss, predictions, labels."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            lbls  = batch["label"].to(device)
            out   = model(input_ids=ids, attention_mask=mask)
            loss  = (criterion(out.logits, lbls) if criterion
                     else model(input_ids=ids, attention_mask=mask, labels=lbls).loss)
            total_loss += loss.item()
            all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    return (f1_score(all_labels, all_preds, average="macro"),
            total_loss / len(loader), all_preds, all_labels)


# ── Shared: get_predictions() ─────────────────────────────
def get_predictions(model, tokenizer, texts, labels, batch_size=32):
    """Run inference on raw text lists, return preds and probs."""
    ds     = HateDataset(texts, labels, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    all_preds, all_probs = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            out   = model(input_ids=batch["input_ids"].to(device),
                          attention_mask=batch["attention_mask"].to(device))
            probs = torch.softmax(out.logits, dim=-1)
            all_preds.extend(probs.argmax(dim=-1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_probs)

In [ ]:
# ── Shared: Obfuscation functions ────────────────────────
LEET_MAP = {'a':'4','e':'3','i':'1','o':'0','s':'5','t':'7','g':'9','b':'8'}

def leet_speak(text, rate=0.4):
    """Replace letters with leet equivalents (e.g. hate → h4t3)."""
    return "".join(LEET_MAP[c.lower()] if c.lower() in LEET_MAP
                   and random.random() < rate else c for c in text)

def insert_punctuation(text, rate=0.3):
    """Insert punctuation between chars of random words (e.g. hate → h.a.t.e)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            words[i] = random.choice([".","-","_","*"]).join(list(w))
    return " ".join(words)

def char_repeat(text, rate=0.3):
    """Randomly repeat a character in random words (e.g. hate → haate)."""
    words = text.split()
    for i, w in enumerate(words):
        if len(w) > 3 and random.random() < rate:
            idx = random.randint(1, len(w)-2)
            words[i] = w[:idx] + w[idx]*random.randint(2,4) + w[idx+1:]
    return " ".join(words)

def combined_obfuscation(text):
    """Apply all three obfuscation strategies."""
    return char_repeat(insert_punctuation(leet_speak(text, 0.3), 0.2), 0.2)

OBFUSCATION_FNS = {
    "leet_speak":  leet_speak,
    "punctuation": insert_punctuation,
    "char_repeat": char_repeat,
    "combined":    combined_obfuscation,
}

# Preview
random.seed(RANDOM_SEED)
s = "I hate those people they should all leave"
print(f"Original    : {s}")
print(f"Leet        : {leet_speak(s)}")
print(f"Punctuation : {insert_punctuation(s)}")
print(f"Char repeat : {char_repeat(s)}")
print(f"Combined    : {combined_obfuscation(s)}")

In [ ]:
# ── Shared: training loop with early stopping ─────────────
def train_model(model, train_loader, val_loader, epochs, lr,
                weight_decay, patience, criterion=None, label=""):
    """
    Fine-tune model with AdamW + linear warmup + early stopping on val loss.
    Returns best model state dict and training history.
    """
    n_steps  = len(train_loader) * epochs
    n_warmup = int(0.1 * n_steps)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_scheduler("linear", optimizer=optimizer,
                               num_warmup_steps=n_warmup, num_training_steps=n_steps)

    history           = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_val_loss     = float("inf")
    best_state        = None
    no_improve        = 0
    stopped_at        = epochs

    print(f"Training {label} | max {epochs} epochs | patience {patience}\n")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader):
            ids, mask, lbls = (batch["input_ids"].to(device),
                               batch["attention_mask"].to(device),
                               batch["label"].to(device))
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls) if criterion else                    model(input_ids=ids, attention_mask=mask, labels=lbls).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
            if (step+1) % 100 == 0:
                print(f"  Ep {epoch+1} | Step {step+1}/{len(train_loader)} | Loss {loss.item():.4f}")

        avg_train = total_loss / len(train_loader)
        val_f1, avg_val, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)
        history["val_f1"].append(val_f1)

        print(f"\n── Epoch {epoch+1}/{epochs} ──")
        print(f"   Train loss : {avg_train:.4f}")
        print(f"   Val loss   : {avg_val:.4f}")
        print(f"   Val F1     : {val_f1:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
            print(f"   ✓ Best model (val_loss={best_val_loss:.4f}, F1={val_f1:.4f})")
        else:
            no_improve += 1
            print(f"   ✗ No improvement ({no_improve}/{patience})")
            if no_improve >= patience:
                stopped_at = epoch + 1
                print(f"\n⚡ Early stopping at epoch {stopped_at}")
                break
        print()

    print(f"Done. Best val_loss={best_val_loss:.4f}")
    return best_state, history, stopped_at


# ── Shared: plot training curves ─────────────────────────
def plot_curves(history, stopped_at=None, title="", save_path="curves.png"):
    n = len(history["train_loss"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(range(1,n+1), history["train_loss"], marker="o", label="Train", color="#457b9d")
    axes[0].plot(range(1,n+1), history["val_loss"],   marker="o", label="Val",   color="#e63946")
    if stopped_at and stopped_at < MAX_EPOCHS:
        axes[0].axvline(stopped_at - PATIENCE, color="gray", linestyle="--", label="Early stop")
    axes[0].set_title(f"Loss per Epoch {title}", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)
    axes[1].plot(range(1,n+1), history["val_f1"], marker="o", color="#2a9d8f")
    axes[1].set_title(f"Val Macro F1 {title}", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
    axes[1].spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved {save_path}")


# ── Shared: confusion matrix plot ────────────────────────
def plot_cm(labels, preds, title="", cmap="Blues", save_path="cm.png"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["hate","offensive","normal"],
                yticklabels=["hate","offensive","normal"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix {title}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved {save_path}")


# ── Shared: robustness evaluation ────────────────────────
def robustness_eval(model, tokenizer, texts, labels, reference_f1=None):
    """Evaluate model on clean + all obfuscated variants. Returns results dict."""
    random.seed(RANDOM_SEED)
    results, obf_cache = {}, {"clean": texts}
    preds_clean, _ = get_predictions(model, tokenizer, texts, labels)
    results["clean"] = f1_score(labels, preds_clean, average="macro")
    ref = reference_f1 or results["clean"]
    print(f"  clean          → F1: {results['clean']:.4f}")
    for name, fn in OBFUSCATION_FNS.items():
        obf = [fn(t) for t in texts]
        obf_cache[name] = obf
        p, _ = get_predictions(model, tokenizer, obf, labels)
        f1   = f1_score(labels, p, average="macro")
        results[name] = f1
        print(f"  {name:<16} → F1: {f1:.4f}  (drop: {ref-f1:+.4f})")
    return results, obf_cache, preds_clean

print("✓ All shared utilities defined")

---
## Task 1 — Problem Definition & Dataset Understanding

**Task:** Classify social media posts into three categories — *hate speech*, *offensive language*, and *normal*.  
This is an NLP classification problem because it requires understanding context, implicit meaning, and social norms from raw text — not just surface-level keywords.

**Dataset:** [HateXplain](https://github.com/hate-alert/HateXplain) — ~20,000 posts from Twitter and Gab, annotated by 3 workers each. Labels are resolved via majority vote. Posts with no majority (~2–3%) are discarded.

**Known limitations:** class imbalance; annotation subjectivity; English-only; no demographic information on annotators.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/"

def download_json(filename):
    with urllib.request.urlopen(BASE_URL + filename) as r:
        return json.loads(r.read().decode())

print("Downloading dataset from GitHub...")
data       = download_json("dataset.json")
split_info = download_json("post_id_divisions.json")
print(f"✓ Total posts: {len(data)} | Splits: {list(split_info.keys())}")

# Inspect one example
sample_id = list(data.keys())[0]
print(f"\n── Sample post ({sample_id}) ──")
print(json.dumps(data[sample_id], indent=2)[:600])

In [ ]:
def majority_vote(label_list):
    count = Counter(label_list)
    top   = count.most_common(1)[0]
    return top[0] if top[1] >= 2 else None

def build_dataframe(post_ids):
    rows = []
    for pid in post_ids:
        if pid not in data: continue
        post   = data[pid]
        labels = [a["label"] for a in post["annotators"]]
        vote   = majority_vote(labels)
        if vote is None: continue
        rows.append({"post_id": pid,
                     "text":       " ".join(post["post_tokens"]),
                     "label":      LABEL_MAP[vote],
                     "label_name": LABEL_NAMES[LABEL_MAP[vote]]})
    return pd.DataFrame(rows)

train_df = build_dataframe(split_info["train"])
val_df   = build_dataframe(split_info["val"])
test_df  = build_dataframe(split_info["test"])

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
print(f"\nLabel distribution (train):\n{train_df['label_name'].value_counts()}")

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)
print("\n✓ Saved train.csv / val.csv / test.csv")

---
## Task 2 — Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

# ── Class distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [("Train",train_df),("Validation",val_df),("Test",test_df)]):
    counts = df["label_name"].value_counts().reindex(["hate","offensive","normal"])
    bars   = ax.bar(counts.index, counts.values,
                    color=[COLORS[l] for l in counts.index], edgecolor="white")
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Class"); ax.set_ylabel("Count")
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                str(v), ha="center", fontsize=10)
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Class Distribution across Splits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_class_distribution.png")

In [ ]:
# ── Text length analysis ──────────────────────────────────
train_df["token_count"] = train_df["text"].apply(lambda x: len(str(x).split()))
train_df["char_count"]  = train_df["text"].apply(lambda x: len(str(x)))

print("── Token count by class ──")
print(train_df.groupby("label_name")["token_count"].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color in COLORS.items():
    axes[0].hist(train_df[train_df["label_name"]==label]["token_count"],
                 bins=40, alpha=0.6, label=label, color=color, edgecolor="none")
axes[0].set_title("Token Count Distribution", fontweight="bold")
axes[0].set_xlabel("Tokens"); axes[0].set_ylabel("Frequency"); axes[0].legend()
axes[0].spines[["top","right"]].set_visible(False)

sns.boxplot(data=train_df, x="label_name", y="token_count", palette=COLORS,
            order=["hate","offensive","normal"], ax=axes[1], hue="label_name")
axes[1].set_title("Token Count Boxplot", fontweight="bold")
axes[1].spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eda_text_length.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_text_length.png")

In [ ]:
# ── Top-20 tokens per class ───────────────────────────────
def get_top_tokens(texts, n=20):
    tokens = []
    for text in texts:
        tokens.extend([w.lower() for w in str(text).split()
                       if w.isalpha() and w.lower() not in STOPWORDS and len(w) > 2])
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, color) in zip(axes, COLORS.items()):
    top = get_top_tokens(train_df[train_df["label_name"]==label]["text"])
    words, counts = zip(*top)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor="none")
    ax.set_title(f"Top 20 Tokens — {label.capitalize()}", fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
plt.suptitle("Most Frequent Tokens by Class (NLTK stopwords removed)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_top_tokens.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_top_tokens.png")

In [ ]:
# ── Word clouds ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, cmap) in zip(axes, {"hate":"Reds","offensive":"Oranges","normal":"Greens"}.items()):
    text  = " ".join(train_df[train_df["label_name"]==label]["text"].tolist())
    clean = " ".join([w for w in text.split()
                      if w.lower() not in STOPWORDS and w.isalpha() and len(w) > 2])
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap=cmap, max_words=100, collocations=False).generate(clean)
    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
    ax.set_title(label.capitalize(), fontsize=13, fontweight="bold")
plt.suptitle("Word Clouds by Class", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_wordclouds.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved eda_wordclouds.png")

In [ ]:
# ── Summary statistics ────────────────────────────────────
print("══════════════════════════════════════════")
print("        EDA SUMMARY — TRAIN SPLIT")
print("══════════════════════════════════════════")
total = len(train_df)
for label in ["hate","offensive","normal"]:
    n   = (train_df["label_name"]==label).sum()
    avg = train_df[train_df["label_name"]==label]["token_count"].mean()
    print(f"  {label:<12} | n={n:>4} ({n/total*100:.1f}%) | avg tokens={avg:.1f}")
print(f"  {'TOTAL':<12} | n={total}")
print("══════════════════════════════════════════")
print(f"  Overall avg tokens : {train_df['token_count'].mean():.1f}")
print(f"  Overall avg chars  : {train_df['char_count'].mean():.1f}")
print(f"  Max token count    : {train_df['token_count'].max()}")
print(f"  Min token count    : {train_df['token_count'].min()}")

---
## Task 3 — Data Preprocessing

Preprocessing is intentionally **minimal**: HateBERT is pretrained on raw social media text, so aggressive normalization would hurt performance. We only replace URLs and @mentions with placeholder tokens, and strip `#` from hashtags.

HateBERT uses **WordPiece** subword tokenization — rare or long words are split into known subword pieces (e.g. `foreigners` → `foreign`, `##ers`). `max_length=128` covers >95% of posts in this dataset.

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "[URL]",  text)
    text = re.sub(r"@\w+",            "[USER]", text)
    text = re.sub(r"#(\w+)",           r"\1",   text)
    text = re.sub(r"\s+",              " ",      text).strip()
    return text

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

print("── Before / After cleaning ──")
for i in range(2):
    print(f"\nOriginal : {train_df['text'].iloc[i]}")
    print(f"Cleaned  : {train_df['clean_text'].iloc[i]}")

In [ ]:
# ── Load tokenizer & check token lengths ─────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer: {MODEL_NAME} | Vocab size: {tokenizer.vocab_size}")

sample = train_df["clean_text"].iloc[0]
print(f"\nExample tokenization:")
print(f"  Text   : {sample}")
print(f"  Tokens : {tokenizer.tokenize(sample)[:20]}")

print("\nComputing token lengths...")
train_df["token_len"] = train_df["clean_text"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True)))

plt.figure(figsize=(8,4))
plt.hist(train_df["token_len"], bins=50, color="#457b9d", edgecolor="none")
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"max_length={MAX_LEN}")
plt.title("Token Length Distribution (train)", fontweight="bold")
plt.xlabel("Token count"); plt.ylabel("Frequency"); plt.legend()
plt.tight_layout()
plt.savefig("preprocessing_token_lengths.png", dpi=150, bbox_inches="tight"); plt.show()

pct = (train_df["token_len"] <= MAX_LEN).mean() * 100
print(f"✓ Posts covered by max_length={MAX_LEN}: {pct:.1f}%")

In [ ]:
# ── Save clean CSVs ───────────────────────────────────────
cols = ["post_id","text","clean_text","label","label_name"]
train_df[cols].to_csv("train_clean.csv", index=False)
val_df[cols].to_csv("val_clean.csv",     index=False)
test_df[cols].to_csv("test_clean.csv",   index=False)
print("✓ Saved train_clean.csv / val_clean.csv / test_clean.csv")

---
## Task 4a — Baseline Training (DistilBERT + Early Stopping)

**Model:** DistilBERT (`distilbert-base-uncased`) — lightweight distilled version of BERT-base, 40% smaller and 60% faster while retaining ~97% of BERT's performance on NLU tasks.

**Training setup:**
- AdamW optimizer, lr=2e-5, weight decay=0.1
- Linear scheduler with 10% warmup
- Early stopping on validation loss (patience=2)

In [ ]:
train_df = pd.read_csv("train_clean.csv")
val_df   = pd.read_csv("val_clean.csv")
test_df  = pd.read_csv("test_clean.csv")

# Datasets & loaders
train_ds_4a = HateDataset(train_df["clean_text"].tolist(), train_df["label"].tolist(), tokenizer)
val_ds_4a   = HateDataset(val_df["clean_text"].tolist(),   val_df["label"].tolist(),   tokenizer)
test_ds_4a  = HateDataset(test_df["clean_text"].tolist(),  test_df["label"].tolist(),  tokenizer)

train_loader_4a = DataLoader(train_ds_4a, batch_size=BATCH_SIZE, shuffle=True)
val_loader_4a   = DataLoader(val_ds_4a,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_4a  = DataLoader(test_ds_4a,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ Train batches: {len(train_loader_4a)} | Val batches: {len(val_loader_4a)}")

# Model
model_4a = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
n_params = sum(p.numel() for p in model_4a.parameters() if p.requires_grad)
print(f"✓ Model: {MODEL_NAME} | Params: {n_params:,}")

In [ ]:
# ── Train ─────────────────────────────────────────────────
best_state_4a, history_4a, stopped_4a = train_model(
    model_4a, train_loader_4a, val_loader_4a,
    epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, label="— DistilBERT baseline"
)
plot_curves(history_4a, stopped_4a, title="— DistilBERT Baseline",
            save_path="training_curves_4a.png")

In [ ]:
# ── Restore best model & evaluate on test set ─────────────
model_4a.load_state_dict(best_state_4a)
_, _, preds_4a_test, labels_4a_test = evaluate(model_4a, test_loader_4a)

print("── Task 4a — Test Set Results ──")
print(classification_report(labels_4a_test, preds_4a_test,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(labels_4a_test, preds_4a_test,
        title="— DistilBERT Baseline", cmap="Blues",
        save_path="cm_4a.png")

# Save
model_4a.save_pretrained(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f"✓ Saved to ./{MODEL_SAVE}/")

---
## Task 4b — Improved Training (Class Weights + Adversarial Augmentation)

Two improvements over the baseline:

1. **Class weights** — the loss penalises errors on minority classes (`hate`, `offensive`) more than `normal`, directly reducing false negatives
2. **Adversarial augmentation** — during training, 50% of `hate` and `offensive` posts are randomly obfuscated (leet-speak, punctuation, char repeat). The model learns to detect hate speech even when characters are disguised

In [ ]:
# ── Class weights ─────────────────────────────────────────
labels_array         = train_df["label"].values
class_weights        = compute_class_weight("balanced", classes=np.array([0,1,2]), y=labels_array)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion_4b         = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {LABEL_NAMES[i]:<12} → {w:.4f}")

# ── Augmented datasets ────────────────────────────────────
train_ds_4b = HateDataset(train_df["clean_text"].tolist(), train_df["label"].tolist(),
                           tokenizer, augment_fn=combined_obfuscation, aug_rate=AUG_RATE)
val_ds_4b   = HateDataset(val_df["clean_text"].tolist(),   val_df["label"].tolist(), tokenizer)
test_ds_4b  = HateDataset(test_df["clean_text"].tolist(),  test_df["label"].tolist(), tokenizer)

train_loader_4b = DataLoader(train_ds_4b, batch_size=BATCH_SIZE, shuffle=True)
val_loader_4b   = DataLoader(val_ds_4b,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_4b  = DataLoader(test_ds_4b,  batch_size=BATCH_SIZE, shuffle=False)

# Fresh model instance
model_4b = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
print(f"✓ Improved datasets and model ready")

In [ ]:
# ── Train ─────────────────────────────────────────────────
best_state_4b, history_4b, stopped_4b = train_model(
    model_4b, train_loader_4b, val_loader_4b,
    epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
    patience=PATIENCE, criterion=criterion_4b,
    label="— DistilBERT improved"
)
plot_curves(history_4b, stopped_4b, title="— DistilBERT Improved",
            save_path="training_curves_4b.png")

In [ ]:
# ── Restore best model & evaluate on test set ─────────────
model_4b.load_state_dict(best_state_4b)
_, _, preds_4b_test, labels_4b_test = evaluate(model_4b, test_loader_4b)

print("── Task 4b — Test Set Results ──")
print(classification_report(labels_4b_test, preds_4b_test,
                             target_names=["hate","offensive","normal"], digits=4))
plot_cm(labels_4b_test, preds_4b_test,
        title="— DistilBERT Improved", cmap="Greens",
        save_path="cm_4b.png")

# Save
model_4b.save_pretrained(MODEL_IMP_SAVE)
tokenizer.save_pretrained(MODEL_IMP_SAVE)
print(f"✓ Saved to ./{MODEL_IMP_SAVE}/")

---
## Task 5 — Evaluation & Robustness Analysis

### 5a — Baseline robustness

In [ ]:
test_texts  = test_df["clean_text"].tolist()
test_labels = test_df["label"].tolist()

print("── Baseline robustness ──")
results_4a, obf_cache, preds_clean_4a = robustness_eval(
    model_4a, tokenizer, test_texts, test_labels)

In [ ]:
# ── Robustness bar chart — baseline ──────────────────────
fig, ax = plt.subplots(figsize=(8,4))
bar_colors = ["#2a9d8f"] + ["#e63946"]*4
bars = ax.bar(results_4a.keys(), results_4a.values(),
              color=bar_colors, edgecolor="white")
ax.axhline(results_4a["clean"], color="gray", linestyle="--", label="Baseline clean")
ax.set_ylim(0,1); ax.set_ylabel("Macro F1")
ax.set_title("Robustness to Obfuscation — Baseline", fontweight="bold")
for bar, v in zip(bars, results_4a.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{v:.3f}", ha="center", fontsize=9)
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("robustness_4a.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved robustness_4a.png")

In [ ]:
# ── Per-class breakdown — baseline ───────────────────────
print("── Per-class F1 breakdown — Baseline ──")
header = f"{'Condition':<18} {'hate':>8} {'offensive':>10} {'normal':>8} {'macro':>8}"
print(header); print("─"*len(header))
for name, txt_list in obf_cache.items():
    p, _ = get_predictions(model_4a, tokenizer, txt_list, test_labels)
    pc   = f1_score(test_labels, p, average=None)
    mac  = f1_score(test_labels, p, average="macro")
    print(f"  {name:<16} {pc[0]:>8.4f} {pc[1]:>10.4f} {pc[2]:>8.4f} {mac:>8.4f}")

### 5b — Improved model robustness

In [ ]:
print("── Improved robustness ──")
results_4b, _, preds_clean_4b = robustness_eval(
    model_4b, tokenizer, test_texts, test_labels,
    reference_f1=results_4a["clean"])  # compare drop vs baseline clean F1

In [ ]:
# ── Side-by-side comparison chart ────────────────────────
keys  = list(results_4a.keys())
x     = np.arange(len(keys))
width = 0.35
fig, ax = plt.subplots(figsize=(10,5))
b1 = ax.bar(x-width/2, [results_4a[k] for k in keys], width,
            label="Baseline", color="#457b9d", alpha=0.85)
b2 = ax.bar(x+width/2, [results_4b[k] for k in keys], width,
            label="Improved",  color="#2a9d8f", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(keys)
ax.set_ylim(0,1); ax.set_ylabel("Macro F1")
ax.set_title("Baseline vs Improved — Robustness to Obfuscation", fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("robustness_comparison.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Saved robustness_comparison.png")

print("\n── Δ Summary ──")
print(f"{'Condition':<18} {'Baseline':>10} {'Improved':>10} {'Δ':>8}")
print("─"*50)
for k in keys:
    delta = results_4b[k] - results_4a[k]
    print(f"  {k:<16} {results_4a[k]:>10.4f} {results_4b[k]:>10.4f} {delta:>+8.4f}")

### 5c — Qualitative failure analysis

In [ ]:
# Find posts correct on clean but wrong after combined obfuscation
random.seed(RANDOM_SEED)
combined_texts = obf_cache["combined"]
preds_combined, _ = get_predictions(model_4a, tokenizer, combined_texts, test_labels)

failures = []
for i, (cp, op, tl) in enumerate(zip(preds_clean_4a, preds_combined, test_labels)):
    if cp == tl and op != tl:
        failures.append({"original":   test_texts[i],
                         "obfuscated": combined_texts[i],
                         "true":       LABEL_NAMES[tl],
                         "pred_clean": LABEL_NAMES[cp],
                         "pred_obf":   LABEL_NAMES[op]})

print(f"✓ Failure cases: {len(failures)} ({len(failures)/len(test_texts)*100:.1f}% of test set)\n")
for i, f in enumerate(random.sample(failures, min(5, len(failures)))):
    print(f"── Example {i+1} ──")
    print(f"  Original   : {f['original'][:100]}")
    print(f"  Obfuscated : {f['obfuscated'][:100]}")
    print(f"  True       : {f['true']}")
    print(f"  Clean pred : {f['pred_clean']}  ✓")
    print(f"  Obf pred   : {f['pred_obf']}  ✗\n")

---
## 💾 Save Models to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, os

os.makedirs(DRIVE_DIR, exist_ok=True)

for src, dst_name in [(MODEL_SAVE, "distilbert_baseline"),
                       (MODEL_IMP_SAVE, "distilbert_improved")]:
    dst = os.path.join(DRIVE_DIR, dst_name)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✓ Saved {src} → {dst}")